In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ChessNet(nn.Module):
    def __init__(self):
        super(ChessNet, self).__init__()
        
        self.conv1 = nn.Conv2d(12, 128, 3, padding=1)
        self.conv2 = nn.Conv2d(128, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 128, 3, padding=1)
        
        # Policy head
        self.policy_conv = nn.Conv2d(128, 2, 1)
        self.policy_fc = nn.Linear(2*8*8, 4672)  # all possible moves
        
        # Value head
        self.value_conv = nn.Conv2d(128, 1, 1)
        self.value_fc1 = nn.Linear(8*8, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        # Policy
        p = F.relu(self.policy_conv(x))
        p = p.view(-1, 2*8*8)
        p = self.policy_fc(p)

        # Value
        v = F.relu(self.value_conv(x))
        v = v.view(-1, 8*8)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))

        return p, v

In [ ]:
import chess
import numpy as np

def self_play(model):
    board = chess.Board()
    game_data = []

    while not board.is_game_over():
        state = board_to_tensor(board)

        policy, value = model(state)
        move = select_move(policy, board)

        game_data.append((state, policy))

        board.push(move)

    result = board.result()

    return game_data, result

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import chess
import numpy as np
import random
from collections import defaultdict

# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BOARD_SHAPE = (12, 8, 8)
ACTION_SIZE = 4672  # max move space

# =========================
# BOARD ENCODING
# =========================
def board_to_tensor(board):
    piece_map = board.piece_map()
    tensor = np.zeros(BOARD_SHAPE, dtype=np.float32)

    piece_to_channel = {
        chess.PAWN: 0,
        chess.KNIGHT: 1,
        chess.BISHOP: 2,
        chess.ROOK: 3,
        chess.QUEEN: 4,
        chess.KING: 5
    }

    for square, piece in piece_map.items():
        row = square // 8
        col = square % 8
        channel = piece_to_channel[piece.piece_type]
        if piece.color == chess.BLACK:
            channel += 6
        tensor[channel][row][col] = 1

    return torch.tensor(tensor, dtype=torch.float32).unsqueeze(0).to(DEVICE)

# =========================
# MOVE ENCODING (simple hash)
# =========================
def move_to_index(move):
    return (move.from_square * 64 + move.to_square) % ACTION_SIZE

def index_to_move(index, board):
    legal_moves = list(board.legal_moves)
    return random.choice(legal_moves)

# =========================
# MODEL
# =========================
class ChessNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(12, 128, 3, padding=1)
        self.conv2 = nn.Conv2d(128, 128, 3, padding=1)

        # policy head
        self.policy_conv = nn.Conv2d(128, 2, 1)
        self.policy_fc = nn.Linear(2*8*8, ACTION_SIZE)

        # value head
        self.value_conv = nn.Conv2d(128, 1, 1)
        self.value_fc1 = nn.Linear(8*8, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        # policy
        p = F.relu(self.policy_conv(x))
        p = p.view(-1, 2*8*8)
        p = self.policy_fc(p)

        # value
        v = F.relu(self.value_conv(x))
        v = v.view(-1, 8*8)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))

        return p, v

# =========================
# MCTS NODE
# =========================
class Node:
    def __init__(self, board):
        self.board = board
        self.children = {}
        self.visits = 0
        self.value = 0

# =========================
# MCTS (simplified)
# =========================
def mcts_search(model, board, simulations=25):
    root = Node(board)

    for _ in range(simulations):
        node = root
        sim_board = board.copy()

        # selection
        while node.children:
            move = random.choice(list(node.children.keys()))
            node = node.children[move]
            sim_board.push(move)

        # expansion
        if not sim_board.is_game_over():
            for move in sim_board.legal_moves:
                node.children[move] = Node(sim_board.copy())

        # evaluation
        state = board_to_tensor(sim_board)
        _, value = model(state)

        value = value.item()

        # backprop
        node.visits += 1
        node.value += value

    # pick best move
    best_move = None
    best_visits = -1

    for move, child in root.children.items():
        if child.visits > best_visits:
            best_visits = child.visits
            best_move = move

    return best_move

# =========================
# SELF PLAY
# =========================
def self_play(model):
    board = chess.Board()
    game_data = []

    while not board.is_game_over():
        state = board_to_tensor(board)

        move = mcts_search(model, board)

        policy_target = torch.zeros(ACTION_SIZE).to(DEVICE)
        policy_target[move_to_index(move)] = 1.0

        game_data.append((state, policy_target))

        board.push(move)

    result = board.result()

    if result == "1-0":
        value = 1
    elif result == "0-1":
        value = -1
    else:
        value = 0

    return [(s, p, value) for (s, p) in game_data]

# =========================
# TRAINING
# =========================
def train(model, optimizer, data):
    model.train()

    for state, target_policy, target_value in data:
        pred_policy, pred_value = model(state)

        target_value = torch.tensor([[target_value]], dtype=torch.float32).to(DEVICE)

        loss_policy = F.cross_entropy(pred_policy, target_policy.unsqueeze(0))
        loss_value = F.mse_loss(pred_value, target_value)

        loss = loss_policy + loss_value

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# =========================
# MAIN LOOP
# =========================
def main():
    model = ChessNet().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for episode in range(50):  # increase later
        data = self_play(model)
        train(model, optimizer, data)

        print(f"Episode {episode} completed")

if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import chess
import numpy as np
import random
import math
from collections import deque

# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ACTION_SIZE = 4672
SIMULATIONS = 50
BUFFER_SIZE = 10000
BATCH_SIZE = 64

# =========================
# BOARD ENCODING
# =========================
def board_to_tensor(board):
    tensor = np.zeros((12, 8, 8), dtype=np.float32)

    for square, piece in board.piece_map().items():
        row = square // 8
        col = square % 8
        channel = piece.piece_type - 1
        if piece.color == chess.BLACK:
            channel += 6
        tensor[channel][row][col] = 1

    return torch.tensor(tensor, dtype=torch.float32).to(DEVICE)

# =========================
# MOVE ENCODING
# =========================
def move_to_index(move):
    return (move.from_square * 64 + move.to_square) % ACTION_SIZE

# =========================
# MODEL
# =========================
class ChessNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(12, 128, 3, padding=1)
        self.conv2 = nn.Conv2d(128, 128, 3, padding=1)

        # policy
        self.policy_conv = nn.Conv2d(128, 2, 1)
        self.policy_fc = nn.Linear(2*8*8, ACTION_SIZE)

        # value
        self.value_conv = nn.Conv2d(128, 1, 1)
        self.value_fc1 = nn.Linear(8*8, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        p = F.relu(self.policy_conv(x))
        p = p.view(-1, 2*8*8)
        p = self.policy_fc(p)

        v = F.relu(self.value_conv(x))
        v = v.view(-1, 8*8)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))

        return p, v

# =========================
# MCTS NODE
# =========================
class Node:
    def __init__(self, board, prior=0):
        self.board = board
        self.prior = prior
        self.children = {}
        self.visits = 0
        self.value = 0

    def value_avg(self):
        return self.value / (self.visits + 1e-5)

# =========================
# UCB
# =========================
def ucb(parent, child, c=1.4):
    return child.value_avg() + c * child.prior * math.sqrt(parent.visits) / (1 + child.visits)

# =========================
# MCTS
# =========================
def mcts(model, board):
    root = Node(board)

    # expand root
    state = board_to_tensor(board).unsqueeze(0)
    policy, _ = model(state)
    policy = F.softmax(policy, dim=1).detach().cpu().numpy()[0]

    for move in board.legal_moves:
        root.children[move] = Node(board.copy(), prior=policy[move_to_index(move)])

    # simulations
    for _ in range(SIMULATIONS):
        node = root
        sim_board = board.copy()
        path = []

        # selection
        while node.children:
            move, node = max(node.children.items(), key=lambda item: ucb(node, item[1]))
            sim_board.push(move)
            path.append(node)

        # evaluation
        if not sim_board.is_game_over():
            state = board_to_tensor(sim_board).unsqueeze(0)
            policy, value = model(state)
            policy = F.softmax(policy, dim=1).detach().cpu().numpy()[0]

            # expand
            for move in sim_board.legal_moves:
                node.children[move] = Node(sim_board.copy(), prior=policy[move_to_index(move)])

            value = value.item()
        else:
            result = sim_board.result()
            value = 1 if result == "1-0" else -1 if result == "0-1" else 0

        # backprop
        for n in path:
            n.visits += 1
            n.value += value

    # policy target
    visits = np.array([child.visits for child in root.children.values()])
    moves = list(root.children.keys())

    probs = visits / np.sum(visits)

    return moves, probs

# =========================
# SELF PLAY
# =========================
def self_play(model):
    board = chess.Board()
    memory = []

    while not board.is_game_over():
        moves, probs = mcts(model, board)

        move = random.choices(moves, weights=probs)[0]

        state = board_to_tensor(board)
        policy_target = torch.zeros(ACTION_SIZE)
        for m, p in zip(moves, probs):
            policy_target[move_to_index(m)] = p

        memory.append((state, policy_target))
        board.push(move)

    result = board.result()
    value = 1 if result == "1-0" else -1 if result == "0-1" else 0

    return [(s, p, value) for (s, p) in memory]

# =========================
# TRAIN
# =========================
def train(model, optimizer, buffer):
    if len(buffer) < BATCH_SIZE:
        return

    batch = random.sample(buffer, BATCH_SIZE)

    states = torch.stack([x[0] for x in batch]).to(DEVICE)
    policies = torch.stack([x[1] for x in batch]).to(DEVICE)
    values = torch.tensor([[x[2]] for x in batch], dtype=torch.float32).to(DEVICE)

    pred_p, pred_v = model(states)

    loss_p = -torch.mean(torch.sum(policies * F.log_softmax(pred_p, dim=1), dim=1))
    loss_v = F.mse_loss(pred_v, values)

    loss = loss_p + loss_v

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# =========================
# EVAL
# =========================
def evaluate(model, games=10):
    wins = 0

    for _ in range(games):
        board = chess.Board()

        while not board.is_game_over():
            if board.turn:
                moves, probs = mcts(model, board)
                move = moves[np.argmax(probs)]
            else:
                move = random.choice(list(board.legal_moves))

            board.push(move)

        if board.result() == "1-0":
            wins += 1

    return wins / games

# =========================
# MAIN
# =========================
def main():
    model = ChessNet().to(DEVICE)

    # LOAD if exists
    try:
        model.load_state_dict(torch.load("latest.pth"))
        print("Loaded previous model")
    except:
        print("Training from scratch")

    optimizer = optim.Adam(model.parameters(), lr=0.0003)
    buffer = deque(maxlen=BUFFER_SIZE)

    for episode in range(200):

        game_data = self_play(model)
        buffer.extend(game_data)

        train(model, optimizer, buffer)

        if episode % 10 == 0:
            win_rate = evaluate(model)
            print(f"Episode {episode} | Win rate vs random: {win_rate}")

            torch.save(model.state_dict(), "latest.pth")

if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import chess
import numpy as np
import random
import math
from collections import deque

DEVICE = torch.device("cpu")   # FORCE CPU

ACTION_SIZE = 4672
SIMULATIONS = 10      # reduced heavily
BUFFER_SIZE = 5000
BATCH_SIZE = 32
MAX_MOVES = 40        # limit game length

# =========================
# BOARD ENCODING
# =========================
def board_to_tensor(board):
    tensor = np.zeros((12, 8, 8), dtype=np.float32)

    for square, piece in board.piece_map().items():
        row = square // 8
        col = square % 8
        channel = piece.piece_type - 1
        if piece.color == chess.BLACK:
            channel += 6
        tensor[channel][row][col] = 1

    return torch.tensor(tensor, dtype=torch.float32)

# =========================
# MOVE ENCODING
# =========================
def move_to_index(move):
    return (move.from_square * 64 + move.to_square) % ACTION_SIZE

# =========================
# MODEL (LIGHTER)
# =========================
class ChessNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(12, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1)

        self.policy_conv = nn.Conv2d(64, 2, 1)
        self.policy_fc = nn.Linear(2*8*8, ACTION_SIZE)

        self.value_conv = nn.Conv2d(64, 1, 1)
        self.value_fc1 = nn.Linear(8*8, 32)
        self.value_fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        p = F.relu(self.policy_conv(x))
        p = p.view(-1, 2*8*8)
        p = self.policy_fc(p)

        v = F.relu(self.value_conv(x))
        v = v.view(-1, 8*8)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))

        return p, v

# =========================
# MCTS NODE
# =========================
class Node:
    def __init__(self, board, prior=0):
        self.board = board
        self.prior = prior
        self.children = {}
        self.visits = 0
        self.value = 0

    def value_avg(self):
        return self.value / (self.visits + 1e-5)

# =========================
# UCB
# =========================
def ucb(parent, child, c=1.4):
    return child.value_avg() + c * child.prior * math.sqrt(parent.visits + 1) / (1 + child.visits)

# =========================
# MCTS (FAST)
# =========================
def mcts(model, board):
    root = Node(board)

    state = board_to_tensor(board).unsqueeze(0)
    policy, _ = model(state)
    policy = F.softmax(policy, dim=1).detach().numpy()[0]

    for move in board.legal_moves:
        root.children[move] = Node(board.copy(), prior=policy[move_to_index(move)])

    for _ in range(SIMULATIONS):
        node = root
        sim_board = board.copy()
        path = []

        while node.children:
            move, node = max(node.children.items(), key=lambda item: ucb(node, item[1]))
            sim_board.push(move)
            path.append(node)

        if not sim_board.is_game_over():
            state = board_to_tensor(sim_board).unsqueeze(0)
            _, value = model(state)
            value = value.item()
        else:
            result = sim_board.result()
            value = 1 if result == "1-0" else -1 if result == "0-1" else 0

        for n in reversed(path):
            n.visits += 1
            n.value += value
            value = -value

    moves = list(root.children.keys())
    visits = np.array([root.children[m].visits for m in moves])
    probs = visits / np.sum(visits)

    return moves, probs

# =========================
# SELF PLAY
# =========================
def self_play(model):
    board = chess.Board()
    memory = []

    for _ in range(MAX_MOVES):
        if board.is_game_over():
            break

        moves, probs = mcts(model, board)

        move = random.choices(moves, weights=probs)[0]

        state = board_to_tensor(board)

        policy_target = torch.zeros(ACTION_SIZE)
        for m, p in zip(moves, probs):
            policy_target[move_to_index(m)] = p

        memory.append((state, policy_target))
        board.push(move)

    result = board.result()
    value = 1 if result == "1-0" else -1 if result == "0-1" else 0

    data = []
    for i, (s, p) in enumerate(memory):
        val = value if i % 2 == 0 else -value
        data.append((s, p, val))

    return data

# =========================
# TRAIN
# =========================
def train(model, optimizer, buffer):
    if len(buffer) < BATCH_SIZE:
        return

    batch = random.sample(buffer, BATCH_SIZE)

    states = torch.stack([x[0] for x in batch])
    policies = torch.stack([x[1] for x in batch])
    values = torch.tensor([[x[2]] for x in batch], dtype=torch.float32)

    pred_p, pred_v = model(states)

    loss_p = -torch.mean(torch.sum(policies * F.log_softmax(pred_p, dim=1), dim=1))
    loss_v = F.mse_loss(pred_v, values)

    loss = loss_p + loss_v

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# =========================
# EVAL
# =========================
def evaluate(model, games=5):
    wins = 0
    for _ in range(games):
        board = chess.Board()
        while not board.is_game_over():
            if board.turn:
                moves, probs = mcts(model, board)
                move = moves[np.argmax(probs)]
            else:
                move = random.choice(list(board.legal_moves))
            board.push(move)

        if board.result() == "1-0":
            wins += 1

    return wins / games

# =========================
# MAIN
# =========================
def main():
    model = ChessNet()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    buffer = deque(maxlen=BUFFER_SIZE)

    for episode in range(300):

        game_data = self_play(model)
        buffer.extend(game_data)

        train(model, optimizer, buffer)

        if episode % 20 == 0:
            win_rate = evaluate(model)
            print(f"Episode {episode} | Win rate: {win_rate}")

if __name__ == "__main__":
    main()

In [ ]:
import chess
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

# 1. Neural Network: A simple CNN that "sees" the board
class ChessNet(nn.Module):
    def __init__(self):
        super(ChessNet, self).__init__()
        self.conv1 = nn.Conv2d(12, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Sequential(nn.Linear(64 * 8 * 8, 256), nn.ReLU(), nn.Linear(256, 1))

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = x.view(-1, 64 * 8 * 8)
        return self.fc1(x)

# 2. Board Encoding: Convert chess board to a 12-layer tensor (6 pieces x 2 colors)
def encode_board(board):
    matrix = np.zeros((12, 8, 8), dtype=np.float32)
    piece_map = {'P':0, 'N':1, 'B':2, 'R':3, 'Q':4, 'K':5,
                 'p':6, 'n':7, 'b':8, 'r':9, 'q':10, 'k':11}
    for square, piece in board.piece_map().items():
        row, col = divmod(square, 8)
        matrix[piece_map[piece.symbol()], row, col] = 1
    return torch.from_numpy(matrix).unsqueeze(0)

# 3. Simple Training Loop (The 30-Minute Version)
def train():
    model = ChessNet()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    for episode in range(50): # 50 quick games
        board = chess.Board()
        while not board.is_game_over():
            legal_moves = list(board.legal_moves)
            move_scores = []
            
            # Evaluate all legal moves
            for move in legal_moves:
                board.push(move)
                score = model(encode_board(board))
                move_scores.append(score)
                board.pop()
            
            # Epsilon-greedy: Choose best move or random
            if random.random() > 0.1:
                move = legal_moves[torch.argmax(torch.cat(move_scores))]
            else:
                move = random.choice(legal_moves)

            # Reward logic: Simplified for speed (Capturing pieces)
            reward = 0
            if board.is_capture(move): reward = 1 
            
            board.push(move)
            
            # Update Model
            target = torch.tensor([[reward]], dtype=torch.float32)
            current_pred = model(encode_board(board))
            loss = loss_fn(current_pred, target)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        print(f"Episode {episode} complete.")

train()